# TF–IDF + Cosine Similarity for Resume–Job Matching

1) Loading the datasets.

In [1]:
from datasets import load_dataset

# Job descriptions dataset
ds = load_dataset("lang-uk/recruitment-dataset-job-descriptions-english")
df_jobs = ds["train"].to_pandas()
df_jobs.to_csv("job-descriptions.csv", index=False)

# For the resume dataset
# Place CareerCorpus.xlsx in the same working directory before running the notebook


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/141897 [00:00<?, ? examples/s]

2) Packages needed for TF–IDF and cosine similarity algorithm testing

In [2]:
!pip install pandas openpyxl scikit-learn

3) Reading the datasets

In [3]:
import pandas as pd
import numpy as np

df_resumes = pd.read_excel("CareerCorpus.xlsx")
df_jobs = pd.read_csv("job-descriptions.csv")

4) Inspecting the datasets

In [4]:
print(df_resumes.head())
print(df_resumes.columns)

print(df_jobs.head())
print(df_jobs.columns)

         ID   Domain                                          Education  \
0  74552449  Banking  Master of Science in International Trade, Univ...   
1  79041971  Banking  High School Diploma, Federal Way Senior High S...   
2  77156708  Banking  Master of Management, Business Management — Co...   
3  24580361  Banking  B.S. in Operations Management, University of D...   
4  34953092  Banking  M.S. Computer Engineering, University of Misso...   

                             Skills and Achievements  \
0  Skilled in cash handling, loan operations, fin...   
1  Strong leadership, team management, and client...   
2  NMLS #1796859; business development; project m...   
3  Microsoft Office (Excel/PowerPoint/Word/Access...   
4  C/C++, Python, MATLAB, SQL, R, LUA, VBA; ML (s...   

                                          Experience      Job_type  \
0  Professional (11/2016–Present)—opened and mana...   Entry-level   
1  Banking Professional (Aug 2013–Present)—overse...     Mid-level   
2 

5) Defining the text for each dataset and putting them together

Resume dataset relevant columns:
*   *Education*
*   *Skills and Achievements*
*   *Experience*
These together form the CV content.

Jobs dataset relevant columns:
*   *Position*
*   *Long Description*

In [5]:
# For the resume dataset
resumes = df_resumes.copy()

resumes["text"] = (
    resumes["Education"].fillna('') + " " +
    resumes["Skills and Achievements"].fillna('') + " " +
    resumes["Experience"].fillna('')
)

resumes = resumes[["text", "Domain", "Job_type"]]

# For the jobs dataset
jobs = df_jobs.copy()

jobs["text"] = (
    jobs["Position"].fillna('') + " " +
    jobs["Long Description"].fillna('')
)

jobs = jobs[["text", "Primary Keyword"]]

# Removing empty rows
resumes = resumes.dropna(subset=["text"]).copy()
jobs = jobs.dropna(subset=["text"]).copy()

6) Minimal cleaning

In [6]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

resumes["cleaned"] = resumes["text"].apply(clean_text)
jobs["cleaned"] = jobs["text"].apply(clean_text)

7) Fiting TF-IDF on both datasets and transforming them into vectors

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

all_texts = list(resumes["cleaned"]) + list(jobs["cleaned"])

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(all_texts)

# For transforming them into vectors
resume_vecs = vectorizer.transform(resumes["cleaned"])
job_vecs = vectorizer.transform(jobs["cleaned"])

8) Cheking similarity

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(job_vecs, resume_vecs)

**Testing** **1**

In [9]:
job_index = 0

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

scores = similarity_matrix[job_index]

resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)

ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
10 + Blockchain Nodes / Masternodes to set up *Requirements*

We're looking for a long term collaboration with someone that has an experience in crypto, masternodes, nodes, validators etc. We need to set up:

Kyber Network
Nebulas
SecretNetwork
Tron
Aion
DeFiChain
EOS
TomoChain
Elrond
IRISnet
IoTeX
Terra
ChainX
Thorchain

Succesful candidates will have an opportunity to get more jobs and long term collaboration.

TOP MATCHES:
                                               preview     score
155  Elementary Teaching Credential, University of ...  0.015602
248  B.S., Florida State University. B2B sales, acc...  0.013602
152  M.Ed., Teaching—Virginia Commonwealth Universi...  0.011780
239  A.A. Business Administration, Mount Olive Coll...  0.010118
79   BBA, Accounting — University of Massachusetts ...  0.010069
204  B.S. Applied Management, University of Califor...  0.009431
297  M.Sc. in CSE (CGPA 3.97/4.0) & B.Sc. in CSE (C...  0.009037
170  B.A. Childhood Education (1–6), Brooklyn

**Testing** **2**

In [10]:
job_index = 6

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

scores = similarity_matrix[job_index]

resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)

ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
1806/03 Sales Manager **Requirements:**
•⠀Experience in IT sales 1+ year
•⠀Generic understanding of IT technologies and Software Development process
•⠀Knowledge of sales techniques, sales markets
•⠀Good communication and negotiation skills
•⠀Fluent written and good spoken English

**We offer:**
•⠀Friendly teams, experienced colleagues, and perfect work equipment
•⠀Opportunities for career growth and raising professional skills
•⠀Competitive compensation and regular results-based salary

TOP MATCHES:
                                               preview     score
218  BSBA in Marketing, University of Nebraska–Linc...  0.144454
75   B.A., Music — Southern University, Baton Rouge...  0.143444
77   BBA, Accounting — Mercer University (1999). F&...  0.123732
234  B.S. Business Administration, Eastern Nazarene...  0.122962
92   Certificate—Finance & Automotive Insurance (20...  0.121873
220  Certificate in Marketing, Temple Real Estate S...  0.116096
99   2 years Medical Engineering/Bu

**Testing** **3**

In [16]:
job_index = 82

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

scores = similarity_matrix[job_index]

resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)

ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
2D-, 3D-Animator position Onlyplay is a fast-growing gaming product development company. Our team develops high-quality games to launch them on popular platforms.

We are inspired by new ideas and bring trends to the gaming world. For the coming years, we face serious goals and many interesting tasks.

With the aim of reaching new horizons in the gaming industry, we are announcing a contest for the position 2D-, 3D-Animator, who will join us in Kyiv on a full-time basis. We are a product com

TOP MATCHES:
                                               preview     score
243  B.F.A. Industrial Design, College of Creative ...  0.035203
287  B.Arch in Architecture, Chittagong Univ. of En...  0.034889
208  BA, Hebert H. Lehman College (2012) with fine ...  0.034819
245  B.S. Business Management & Accounting, UVT Col...  0.027596
154  B.A., African American Studies—The Ohio State ...  0.026667
204  B.S. Applied Management, University of Califor...  0.025834
274  M.Sc. in Computer Scienc

**Testing** **4**

In [13]:
job_index = 24

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

scores = similarity_matrix[job_index]

resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)

ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
1.Middle/Senior Embedded C/C++ The CHI team is looking for a Middle/Senior Embedded C/C++

About the client:
The leading global supplier of embedded and connected software for the automotive industry.
Client`s company develops the award-winning iGO Navigation Engine, which is flexible, designed for automotive use, and assists in prototyping and developing proofs of concept, whether it’s navigation, connectivity, HMI, or the complete in-car experience.
For deep integration, he configures his t

TOP MATCHES:
                                               preview     score
1    High School Diploma, Federal Way Senior High S...  0.050540
278  Ph.D. in Computer Science, Louisiana Tech Univ...  0.041725
21   B.S., Finance – University of Nevada, Las Vega...  0.038512
36   BPS, Computer Information Systems – DeVry Inst...  0.034980
257  B.Sc. in Computer Science & Engineering (2020–...  0.032174
151  M.Ed., Elementary Education & Teaching—UCLA (2...  0.030636
24   B.B.A., Business Financ

**Testing** **5** (with custom job description)

In [11]:
custom_job = """
Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.
"""
#Cleaning
custom_clean = clean_text(custom_job)
#Transform into a vector
custom_vec = vectorizer.transform([custom_clean])
#Similarity
scores = cosine_similarity(custom_vec, resume_vecs)[0]

#And ranking
resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)

ranked["preview"] = ranked["text"].str[:300]

print("CUSTOM JOB:")
print(custom_job)

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

CUSTOM JOB:

Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.


TOP MATCHES:
                                               preview     score
249  B.S. in Information Systems, Northeastern Univ...  0.097093
40   Bachelors in Marketing, University of North Al...  0.092630
235  B.S. Metallurgy & Materials Engineering, Color...  0.072726
20   B.S., Finance – Arizona State University, W.P....  0.069655
287  B.Arch in Architecture, Chittagong Univ. of En...  0.062323
291  M.Sc. in EEE (2025–26) & B.Sc. in EEE (2019–24...  0.058649
70   Certified Project Manager, Stanford (2014); B....  0.056639
242  Diploma, Fashion Designer, Richard Robinson Ac...  0.055418
254  B.Sc. in CSE, East West University (2017–2022)...  0.054405
280  M.Sc. in Food Processing & Preservation (2025–...  0.050220


10) Saving the rankings for the tested dataset jobs and the custom job

In [18]:
all_results = []

# for dataset jobs
tested_jobs = [0, 6, 82, 24]
k = 10

for job_index in tested_jobs:
    scores = similarity_matrix[job_index]
    top_k_idx = np.argsort(scores)[-k:][::-1]

    for rank, idx in enumerate(top_k_idx, start=1):
        all_results.append({
            "job_id": f"dataset_{job_index}",
            "job_text": jobs.iloc[job_index]["text"][:300],
            "rank": rank,
            "resume_index": idx,
            "score": scores[idx],
            "resume_domain": resumes.iloc[idx]["Domain"]
        })

# for custom job
custom_scores = cosine_similarity(custom_vec, resume_vecs)[0]
top_k_idx = np.argsort(custom_scores)[-k:][::-1]

for rank, idx in enumerate(top_k_idx, start=1):
    all_results.append({
        "job_id": "custom_0",
        "job_text": custom_job[:300],
        "rank": rank,
        "resume_index": idx,
        "score": custom_scores[idx],
        "resume_domain": resumes.iloc[idx]["Domain"]
    })

final_df = pd.DataFrame(all_results)
final_df.to_csv("TD-IDF_TestingResults.csv", index=False)
final_df.head(20)

,job_id,job_text,rank,resume_index,score,resume_domain
0,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,1,155,0.015602,TEACHER
1,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,2,248,0.013602,Apparel
2,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,3,152,0.011780,TEACHER
3,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,4,239,0.010118,Apparel
4,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,5,79,0.010069,Finance
5,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,6,204,0.009431,Apparel
6,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,7,297,0.009037,Research Assistant
7,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,8,170,0.008849,TEACHER
8,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,9,130,0.008588,ACCOUNTANT
9,dataset_0,10 + Blockchain Nodes / Masternodes to set up ...,10,171,0.008036,TEACHER
